In [ ]:
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import scipy
import xskillscore as xs
import numpy as np
import dask
from dask.distributed import Client

import albedo_functions as af

In [ ]:
def p_cal(ctrl, sens):
    a = ctrl.resample(time='YE').mean().var(dim='time')
    b = sens.resample(time='YE').mean().var(dim='time')

# Add a small constant to avoid division by zero
    epsilon = 1e-8
    a += epsilon
    b += epsilon

    F = a / b
    #df = len(ctrl.resample(time='YE').mean())
    df = ctrl.resample(time='YE').mean().sizes['time']
    p_value = scipy.stats.f.cdf(F, df, df)
    p_value = xr.Dataset({
        'p': xr.DataArray(p_value, dims=('lat', 'lon'),
                          coords={'lon': a['lon'],'lat': a['lat']})
    })
    return p_value

In [ ]:
def interannual_std(dset, n_bootstrap):
    """
    Calculate interannual standard deviation and its significance via bootstrapping.
    
    Parameters:
        dset (xarray.Dataset): Dataset containing 'time' dimension.
        varname (str): Variable name inside the dataset.
        n_bootstrap (int): Number of bootstrap iterations.
    
    Returns:
        xr.DataArray: observed_std
    """
    # Step 1: Group by year
    annual_mean = dset.groupby('time.year').mean('time')
    
    # Step 2: Observed std dev
    observed_std = annual_mean.std('year')
    
    # Step 3: Bootstrap resampling using xskillscore
    resampled = xs.resampling.resample_iterations_idx(
        annual_mean, n_bootstrap, dim='year', replace=True, dim_max=annual_mean.year.size
    )
    boot_std = resampled.std(dim='year')
    return observed_std

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
COV_PATH = f'{POST_DATA}/'
LAI_PATH = f'{CONFESS_DATA}/'
SAVE_PATH = f'{WORK_DIR}/'

VEGETATION COVERS

In [ ]:
# Open datasets with Dask chunking
cvh_ctrl = xr.open_dataset(COV_PATH + f'{exp_ctrl}_effective_cvh_1x1.nc', chunks={'time': 12})['cvh']
cvl_ctrl = xr.open_dataset(COV_PATH + f'{exp_ctrl}_effective_cvl_1x1.nc', chunks={'time': 12})['cvl']
cvh_sens = xr.open_dataset(COV_PATH + f'{exp_sens}_effective_cvh_199311-201910_1x1.nc', chunks={'time': 12})['cvh']
cvl_sens = xr.open_dataset(COV_PATH + f'{exp_sens}_effective_cvl_199311-201910_1x1.nc', chunks={'time': 12})['cvl']

# select time from 1999
start_date = '1999-11-01'

cvh_ctrl = cvh_ctrl.sel(time=slice(start_date, None))
cvl_ctrl = cvl_ctrl.sel(time=slice(start_date, None))
cvh_sens = cvh_sens.sel(time=slice(start_date, None))
cvl_sens = cvl_sens.sel(time=slice(start_date, None))

# Filter: remove low standard deviation data
cvh_ctrl = xr.where(cvh_ctrl.std('time') >= 0.00005, cvh_ctrl, np.nan)
cvh_sens = xr.where(cvh_sens.std('time') >= 0.00005, cvh_sens, np.nan)
cvl_ctrl = xr.where(cvl_ctrl.std('time') >= 0.00005, cvl_ctrl, np.nan)
cvl_sens = xr.where(cvl_sens.std('time') >= 0.00005, cvl_sens, np.nan)

In [ ]:
# Compute interannual stds
interannual_cvh_ctrl_std = af.land_seas_mask(interannual_std(cvh_ctrl, 1000))
interannual_cvh_sens_std = af.land_seas_mask(interannual_std(cvh_sens, 1000))
interannual_cvl_ctrl_std = af.land_seas_mask(interannual_std(cvl_ctrl, 1000))
interannual_cvl_sens_std = af.land_seas_mask(interannual_std(cvl_sens, 1000))

In [ ]:
# Compute p-values
p_value_cvh = af.land_seas_mask(p_cal(cvh_ctrl, cvh_sens))
p_value_cvl = af.land_seas_mask(p_cal(cvl_ctrl, cvl_sens))

In [ ]:
interannual_cvh_sens_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a52o_cvh_std_1999.nc', compute=True)
interannual_cvh_ctrl_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a1ua_cvh_std_1999.nc', compute=True)
interannual_cvl_sens_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a52o_cvl_std_1999.nc', compute=True)
interannual_cvl_ctrl_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a1ua_cvl_std_1999.nc', compute=True)
p_value_cvh.to_netcdf(f'{SAVE_PATH}delta_cvh_std_p_1999.nc', compute=True)
p_value_cvl.to_netcdf(f'{SAVE_PATH}delta_cvl_std_p_1999.nc', compute=True)

LEAF AREA INDEX

In [ ]:
lai_hv_a52o = xr.open_dataset(LAI_PATH + f'{exp_sens}/original_resolution/lai_hv/{exp_sens}_lai_hv_199311-201910_1x1.nc', chunks=-1)['LAI_HV']
lai_lv_a52o = xr.open_dataset(LAI_PATH + f'{exp_sens}/original_resolution/lai_lv/{exp_sens}_lai_lv_199311-201910_1x1.nc', chunks=-1)['LAI_LV']
# Filter: remove low standard deviation data
lai_hv_a52o = xr.where(lai_hv_a52o.std('time') >= 0.0005, lai_hv_a52o, np.nan)
lai_lv_a52o= xr.where(lai_lv_a52o.std('time') >= 0.0005, lai_lv_a52o, np.nan)
a1ua_land_file = f'{BC_DATA}/'+exp_ctrl+'/icmcl_a1ua_regular_1x1.nc'
dset_a1ua = xr.open_dataset(a1ua_land_file, chunks=-1)

lai_hv_a1ua = dset_a1ua['lai_hv'].sel(time=slice(lai_hv_a52o.time[0],lai_hv_a52o.time[-1]))
lai_lv_a1ua = dset_a1ua['lai_lv'].sel(time=slice(lai_hv_a52o.time[0],lai_hv_a52o.time[-1]))
# Filter: remove low standard deviation data
lai_hv_a1ua = xr.where(lai_hv_a1ua.std('time') >= 0.0005, lai_hv_a1ua, np.nan)
lai_lv_a1ua= xr.where(lai_lv_a1ua.std('time') >= 0.0005, lai_lv_a1ua, np.nan)

# select time from 1999
start_date = '1999-11-01'

laih_ctrl = lai_hv_a1ua.sel(time=slice(start_date, None))
lail_ctrl = lai_lv_a1ua.sel(time=slice(start_date, None))
laih_sens = lai_hv_a52o.sel(time=slice(start_date, None))
lail_sens = lai_lv_a52o.sel(time=slice(start_date, None))

In [ ]:
# Compute interannual stds
interannual_laih_ctrl_std = af.land_seas_mask(interannual_std(laih_ctrl, 1000))
interannual_laih_sens_std = af.land_seas_mask(interannual_std(laih_sens, 1000))
interannual_lail_ctrl_std = af.land_seas_mask(interannual_std(lail_ctrl, 1000))
interannual_lail_sens_std = af.land_seas_mask(interannual_std(lail_sens, 1000))

In [ ]:
# Compute p-values
p_value_laih = af.land_seas_mask(p_cal(laih_ctrl, laih_sens))
p_value_lail = af.land_seas_mask(p_cal(lail_ctrl, lail_sens))

In [ ]:
interannual_laih_sens_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a52o_laih_std_1999.nc', compute=True)
interannual_laih_ctrl_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a1ua_laih_std_1999.nc', compute=True)
interannual_lail_sens_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a52o_lail_std_1999.nc', compute=True)
interannual_lail_ctrl_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a1ua_lail_std_1999.nc', compute=True)
p_value_laih.to_netcdf(f'{SAVE_PATH}delta_laih_std_p_1999.nc', compute=True)
p_value_lail.to_netcdf(f'{SAVE_PATH}delta_lail_std_p_1999.nc', compute=True)

EFFECTIVE LAI

In [ ]:
laih_ctrl = laih_ctrl * cvh_ctrl
lail_ctrl = lail_ctrl * cvl_ctrl
laih_sens = laih_sens * cvh_sens
lail_sens = lail_sens * cvl_sens

In [ ]:
# Compute interannual stds
interannual_laih_ctrl_std = af.land_seas_mask(interannual_std(laih_ctrl, 1000))
interannual_laih_sens_std = af.land_seas_mask(interannual_std(laih_sens, 1000))
interannual_lail_ctrl_std = af.land_seas_mask(interannual_std(lail_ctrl, 1000))
interannual_lail_sens_std = af.land_seas_mask(interannual_std(lail_sens, 1000))

In [ ]:
# Compute p-values
p_value_laih = af.land_seas_mask(p_cal(laih_ctrl, laih_sens))
p_value_lail = af.land_seas_mask(p_cal(lail_ctrl, lail_sens))

In [ ]:
interannual_laih_sens_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a52o_effective_laih_std_1999.nc', compute=True)
interannual_laih_ctrl_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a1ua_effective_laih_std_1999.nc', compute=True)
interannual_lail_sens_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a52o_effective_lail_std_1999.nc', compute=True)
interannual_lail_ctrl_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a1ua_effective_lail_std_1999.nc', compute=True)
p_value_laih.to_netcdf(f'{SAVE_PATH}delta_effective_laih_std_p_1999.nc', compute=True)
p_value_lail.to_netcdf(f'{SAVE_PATH}delta_effective_lail_std_p_1999.nc', compute=True)